In [3]:
%load_ext autoreload
%autoreload 2

import os
# os.environ["CUDA_VISIBLE_DEVICES"] = ""


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [4]:
# ── Standard library ─────────────────────────────────────────────────────────
import copy
import json
import time
from pathlib import Path
from typing import Optional, Tuple

# ── Numerics ─────────────────────────────────────────────────────────────────
import numpy as np

# ── PyTorch ───────────────────────────────────────────────────────────────────
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import Tensor
from torch.export import Dim

# ── ONNX ─────────────────────────────────────────────────────────────────────
import onnx
import onnxruntime as ort

# ── Kokoro TTS ────────────────────────────────────────────────────────────────
from kokoro import KModel, KPipeline
from kokoro.model import KModelForONNX

torch.set_num_threads(8)
print('Imports OK')


Imports OK


In [5]:
# ── Model paths ───────────────────────────────────────────────────────────────
CONFIG_FILE     = 'checkpoints/config.json'
CHECKPOINT_PATH = 'checkpoints/kokoro-v1_0.pth'
VOICE_PATH      = 'checkpoints/voices/af_heart.pt'

# ── Export directory ──────────────────────────────────────────────────────────
EXPORT_DIR = Path('onnx_streaming_vocos')
EXPORT_DIR.mkdir(exist_ok=True)

# ── Inference constants ────────────────────────────────────────────────────────
MAX_INPUT_LENGTH = 510
OPSET_VERSION    = 19

# ── Sample text ───────────────────────────────────────────────────────────────
TEXT = (
    'I had returned to civil practice and had finally abandoned Holmes in his '
    'Baker Street rooms, although I continually visited him and occasionally '
    'even persuaded him to forgo his Bohemian habits so far as to come and '
    'visit us.'
)

def _t() -> float:
    return time.perf_counter()

def to_numpy(t: Tensor) -> np.ndarray:
    t = t.detach().cpu()
    if t.is_floating_point():
        t = t.float()
    return t.numpy()

print(f'Export dir: {EXPORT_DIR.resolve()}')


Export dir: /rhome/eingerman/Projects/DeepLearning/TTS/Kokoro/onnx_streaming_vocos


In [6]:
# ── Load Kokoro model ─────────────────────────────────────────────────────────
with open(CONFIG_FILE) as f:
    config = json.load(f)

kmodel = KModel(config=CONFIG_FILE, model=CHECKPOINT_PATH, disable_complex=True).to('cpu')
model  = KModelForONNX(kmodel).eval()
print('Kokoro model loaded')


/rhome/eingerman/Projects/DeepLearning/TTS/Kokoro/.venv/lib/python3.12/site-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


Kokoro model loaded


In [7]:
# ── Input preparation helpers ─────────────────────────────────────────────────

def load_input_ids(pipeline: KPipeline, text: str):
    if pipeline.lang_code in 'ab':
        _, tokens = pipeline.g2p(text)
        for gs, ps, tks in pipeline.en_tokenize(tokens):
            if not ps:
                continue
    else:
        ps, _ = pipeline.g2p(text)
    if len(ps) > 510:
        ps = ps[:510]
    input_ids = [i for i in (pipeline.model.vocab.get(p) for p in ps) if i is not None]
    input_ids = torch.LongTensor([[0, *input_ids, 0]]).to(pipeline.model.device)
    return ps, input_ids


def load_voice(pipeline: KPipeline, voice: str, phonemes):
    pack = pipeline.load_voice(voice).to('cpu')
    return pack[len(phonemes) - 1]


def load_sample(model: KModelForONNX, text: str = TEXT, voice: str = VOICE_PATH):
    pipeline = KPipeline(lang_code='a', model=model.kmodel, device='cpu')
    phonemes, input_ids = load_input_ids(pipeline, text)
    style = load_voice(pipeline, voice, phonemes)
    speed = torch.IntTensor([1])
    return input_ids, style, speed


def build_padded_inputs(input_ids: Tensor):
    text_mask = torch.zeros(1, MAX_INPUT_LENGTH, dtype=torch.float32)
    text_mask[0, : input_ids.shape[1]] = 1
    input_ids = F.pad(input_ids, (0, MAX_INPUT_LENGTH - input_ids.shape[1]))
    return input_ids, text_mask


print('Helpers defined')


Helpers defined


In [8]:
# ═══════════════════════════════════════════════════════════════════════════════
#  Module definitions — pipeline stages that produce the `en` / `style` inputs
#  needed by F0NPredictorModule (stages 1–4 of the Kokoro pipeline)
# ═══════════════════════════════════════════════════════════════════════════════

class BertEncoderModule(nn.Module):
    """Stage 1: input_ids [1,510], text_mask [1,510] → d_en [1,512,510]"""
    def __init__(self, kmodel):
        super().__init__()
        self.bert         = kmodel.bert
        self.bert_encoder = kmodel.bert_encoder
    def forward(self, input_ids: Tensor, text_mask: Tensor) -> Tensor:
        bert_dur = self.bert(input_ids, attention_mask=text_mask)
        return self.bert_encoder(bert_dur).transpose(-1, -2)


class DurationPredictorCore(nn.Module):
    """Stage 2: d_en, style, text_mask, speed, input_ids → pred_dur, d_enc, t_en"""
    def __init__(self, kmodel):
        super().__init__()
        self.predictor    = kmodel.predictor
        self.text_encoder = kmodel.text_encoder
    def forward(self, d_en, style, text_mask, speed, input_ids):
        d = self.predictor.text_encoder(d_en, style[:, 128:], text_mask)
        x, _ = self.predictor.lstm(d)
        duration = self.predictor.duration_proj(x)
        duration = text_mask * torch.sigmoid(duration).sum(axis=-1) / speed.float()
        pred_dur = torch.round(duration).squeeze()
        d_enc    = d.transpose(-1, -2)
        t_en     = self.text_encoder(input_ids, text_mask)
        return pred_dur, d_enc, t_en


def expand_durations(pred_dur: Tensor) -> Tensor:
    """Python helper (not exported): pred_dur [S] → expanded_indices [T_acoustic]"""
    boundaries = torch.cumsum(pred_dur, dim=0)
    T_total    = int(boundaries[-1].item())
    values     = torch.arange(T_total, device=pred_dur.device)
    return torch.sum(boundaries.unsqueeze(1) <= values.unsqueeze(0), dim=0).long()


class AcousticExpandModule(nn.Module):
    """Stage 3: d_enc_expanded [1,h,T] → en [1,T,512] via shared BiLSTM"""
    def __init__(self, kmodel):
        super().__init__()
        self.shared = kmodel.predictor.shared   # BiLSTM(640,256)
    def forward(self, d_enc_expanded: Tensor) -> Tensor:
        en, _ = self.shared(d_enc_expanded.transpose(-1, -2))
        return en


class F0NPredictorModule(nn.Module):
    """Stage 4 (original): en [1,T,512], style [1,256] → F0_pred [1,T], N_pred [1,T]"""
    def __init__(self, kmodel):
        super().__init__()
        self.predictor = kmodel.predictor
    def forward(self, en: Tensor, style: Tensor):
        return self.predictor.F0Ntrain(en, style[:, 128:256])


print('Module definitions OK')


Module definitions OK


In [9]:
# ═══════════════════════════════════════════════════════════════════════════════
#  PyTorch reference pipeline — produce `en` and `style` for f0n export/testing
# ═══════════════════════════════════════════════════════════════════════════════

bert_module     = BertEncoderModule(model.kmodel).eval()
duration_module = DurationPredictorCore(model.kmodel).eval()
acoustic_module = AcousticExpandModule(model.kmodel).eval()
f0n_module      = F0NPredictorModule(model.kmodel).eval()

raw_input_ids, style, speed = load_sample(model)
input_ids, text_mask = build_padded_inputs(raw_input_ids)

with torch.no_grad():
    t0 = _t()
    d_en = bert_module(input_ids, text_mask)                              # [1,512,510]
    print(f'd_en:           {tuple(d_en.shape)}   {(_t()-t0)*1e3:.0f} ms')

    t0 = _t()
    pred_dur, d_enc, t_en_static = duration_module(
        d_en, style, text_mask, speed, input_ids)
    print(f'pred_dur:       {tuple(pred_dur.shape)}  {(_t()-t0)*1e3:.0f} ms')

    t0 = _t()
    expanded_indices = expand_durations(pred_dur)
    d_enc_exp = torch.index_select(d_enc, 2, expanded_indices)            # [1,h,T_ac]
    print(f'd_enc_exp:      {tuple(d_enc_exp.shape)}  (expand_durations is pure Python)')

    t0 = _t()
    en = acoustic_module(d_enc_exp)                                       # [1,T_ac,512]
    print(f'en:             {tuple(en.shape)}   {(_t()-t0)*1e3:.0f} ms')

    t0 = _t()
    F0_pred, N_pred = f0n_module(en, style)
    print(f'F0_pred/N_pred: {tuple(F0_pred.shape)}  {(_t()-t0)*1e3:.0f} ms')

T_acoustic = en.shape[1]
print(f'\nT_acoustic = {T_acoustic}')
print(f'style      = {tuple(style.shape)}')


d_en:           (1, 512, 510)   234 ms
pred_dur:       (510,)  66 ms
d_enc_exp:      (1, 640, 542)  (expand_durations is pure Python)
en:             (1, 542, 512)   14 ms
F0_pred/N_pred: (1, 1084)  30 ms

T_acoustic = 542
style      = (1, 256)


In [10]:
# ═══════════════════════════════════════════════════════════════════════════════
#  ONNX Export — f0n_predictor
# ═══════════════════════════════════════════════════════════════════════════════

onnx_f0n_path = EXPORT_DIR / 'f0n_predictor.onnx'

with torch.no_grad():
    torch.onnx.export(
        f0n_module,
        args=(en, style),
        f=str(onnx_f0n_path),
        export_params=True,
        input_names=['en', 'style'],
        output_names=['F0_pred', 'N_pred'],
        opset_version=OPSET_VERSION,
        dynamic_axes={
            'en':     {1: 'T_acoustic'},
            'F0_pred':{1: 'T_f0'},
            'N_pred': {1: 'T_f0'},
        },
        do_constant_folding=True,
        dynamo=False,
    )

m = onnx.load(str(onnx_f0n_path))
onnx.checker.check_model(m)
print(f'✓ {onnx_f0n_path.name}  ({onnx_f0n_path.stat().st_size/1e6:.2f} MB)')


/tmp/ipykernel_2449000/3285183368.py:8: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter will be the default. To switch now, set dynamo=True in torch.onnx.export. This new exporter supports features like exporting LLMs with DynamicCache. We encourage you to try it and share feedback to help improve the experience. Learn more about the new export logic: https://pytorch.org/docs/stable/onnx_dynamo.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html.
  torch.onnx.export(
/rhome/eingerman/Projects/DeepLearning/TTS/Kokoro/kokoro/istftnet.py:445: TracerWarning: torch.tensor results are registered as constants in the trace. You can safely ignore this warning if you use this function to create tensors out of constant variables that would be the same every time you call this function. In any other case, this might cause the trace 

✓ f0n_predictor.onnx  (26.37 MB)


In [11]:
# ═══════════════════════════════════════════════════════════════════════════════
#  ONNX Inference — compare F0 / N to PyTorch reference
# ═══════════════════════════════════════════════════════════════════════════════

so = ort.SessionOptions()
so.intra_op_num_threads = 4
f0n_sess = ort.InferenceSession(str(onnx_f0n_path), sess_options=so)

F0_onnx, N_onnx = f0n_sess.run(None, {
    'en':    to_numpy(en),
    'style': to_numpy(style),
})

f0_diff = np.abs(to_numpy(F0_pred) - F0_onnx).max()
n_diff  = np.abs(to_numpy(N_pred)  - N_onnx).max()
print('ONNX vs PyTorch max diff:')
print(f'  F0_pred: {f0_diff:.2e}')
print(f'  N_pred:  {n_diff:.2e}')
print(f'  shapes → F0: {F0_onnx.shape}  N: {N_onnx.shape}')


ONNX vs PyTorch max diff:
  F0_pred: 2.44e-04
  N_pred:  6.68e-06
  shapes → F0: (1, 1084)  N: (1, 1084)


In [12]:
# ═══════════════════════════════════════════════════════════════════════════════
#  LiteRT Export Helpers — _SafeInstanceNorm1d  +  F0NPredictorModuleLT
# ═══════════════════════════════════════════════════════════════════════════════
#
#  Why a separate module variant?
#
#  Problem A — nn.InstanceNorm1d with dynamic T:
#    PyTorch's InstanceNorm1d performs view([N*C, T]) internally.  When T is a
#    symbolic dimension JAX/XLA rejects the reshape.  F.group_norm has the same
#    issue.  Solution: manual mean / (diff*diff).mean avoids any reshape.
#
#  Problem B — internal transpose in F0Ntrain:
#    F0NPredictorModule.forward calls `predictor.F0Ntrain(en, s)` which does:
#        F0 = en.transpose(-1, -2)   # [1,T,512] → [1,512,T]
#    This transpose creates a NEW symbolic axis on axis 2 that is disconnected
#    from the Dim annotation placed on axis 1 of `en` at the module boundary.
#    Subsequent ops then see [1,512,-inf] shapes and fail.
#    Solution: F0NPredictorModuleLT takes en_t=[1,512,T] directly (caller
#    pre-transposes).  T stays on axis 2 throughout and the single Dim
#    annotation on axis 2 propagates correctly through all ops.
#
#  Problem C — var op:
#    Earlier attempt used x.var(dim=2, ...) which maps to aten.var.correction.
#    XLA propagates -inf through the variance, corrupting output shapes.
#    Solution: use (diff*diff).mean(dim=2, ...) instead.

class _SafeInstanceNorm1d(nn.Module):
    """Drop-in InstanceNorm1d that survives LiteRT/JAX tracing.

    Implements instance normalisation manually over axis 2 (T) using only
    mean and element-wise ops — no view/reshape, no group_norm, no aten.var.
    """
    def __init__(self, src: nn.InstanceNorm1d):
        super().__init__()
        self.eps    = src.eps
        self.affine = src.affine
        if src.affine:
            self.weight = nn.Parameter(src.weight.detach().clone())
            self.bias   = nn.Parameter(src.bias.detach().clone())
        else:
            self.weight = None
            self.bias   = None

    def forward(self, x: Tensor) -> Tensor:
        # x: [N, C, T] — normalise each (N,C) instance over T (axis 2)
        mean = x.mean(dim=2, keepdim=True)              # [N, C, 1]
        diff = x - mean                                 # [N, C, T]
        var  = (diff * diff).mean(dim=2, keepdim=True)  # [N, C, 1]
        x    = diff * (var + self.eps).rsqrt()          # [N, C, T]
        if self.affine:
            x = x * self.weight[None, :, None] + self.bias[None, :, None]
        return x


def _patch_instance_norms(module: nn.Module) -> nn.Module:
    """Recursively replace all nn.InstanceNorm1d with _SafeInstanceNorm1d."""
    for name, child in list(module.named_children()):
        if isinstance(child, nn.InstanceNorm1d):
            setattr(module, name, _SafeInstanceNorm1d(child))
        else:
            _patch_instance_norms(child)
    return module


class F0NPredictorModuleLT(nn.Module):
    """LiteRT-exportable F0/N predictor — C-first layout, T on axis 2.

    Inputs:
        en_t   [1, 512, T_acoustic]  — caller pre-transposes: en.transpose(-1,-2)
        style  [1, 256]
    Outputs:
        F0_pred [1, T_f0]  (T_f0 = 2*T_acoustic after upsample in AdainResBlk1d)
        N_pred  [1, T_f0]
    """
    def __init__(self, kmodel):
        super().__init__()
        self.F0      = kmodel.predictor.F0
        self.N       = kmodel.predictor.N
        self.F0_proj = kmodel.predictor.F0_proj
        self.N_proj  = kmodel.predictor.N_proj

    def forward(self, en_t: Tensor, style: Tensor):
        s  = style[:, 128:256]
        F0 = en_t
        for block in self.F0:
            F0 = block(F0, s)
        F0 = self.F0_proj(F0)

        N = en_t
        for block in self.N:
            N = block(N, s)
        N = self.N_proj(N)

        return F0.squeeze(1), N.squeeze(1)


# ── Build, patch, and verify ──────────────────────────────────────────────────
f0n_lt_module = F0NPredictorModuleLT(kmodel)
_patch_instance_norms(f0n_lt_module)
f0n_lt_module.eval()

en_t = en.transpose(-1, -2).float()   # [1, 512, T_acoustic]

with torch.no_grad():
    F0_lt, N_lt = f0n_lt_module(en_t, style)

f0_lt_diff = (F0_pred - F0_lt).abs().max().item()
n_lt_diff  = (N_pred  - N_lt ).abs().max().item()
print('F0NPredictorModuleLT vs original F0NPredictorModule:')
print(f'  F0 max diff: {f0_lt_diff:.2e}   (expect < 1e-4)')
print(f'  N  max diff: {n_lt_diff:.2e}   (expect < 1e-4)')
print(f'  en_t shape : {tuple(en_t.shape)}')
assert f0_lt_diff < 1e-4 and n_lt_diff < 1e-4, 'Numerics mismatch — check C-first layout!'
print('✓ Numerics match')


F0NPredictorModuleLT vs original F0NPredictorModule:
  F0 max diff: 6.10e-05   (expect < 1e-4)
  N  max diff: 1.91e-06   (expect < 1e-4)
  en_t shape : (1, 512, 542)
✓ Numerics match


In [13]:
# ═══════════════════════════════════════════════════════════════════════════════
#  LiteRT Export — F0NPredictorModuleLT via litert_torch
# ═══════════════════════════════════════════════════════════════════════════════
#
#  dynamic_shapes notes:
#    • Use list form (positional), NOT dict — dict form is rejected by current
#      litert_torch.convert().
#    • Every dynamic axis MUST have both min= and max= bounds.  Unbounded dims
#      propagate as -inf through XLA shape inference and fail.
#    • Annotate T on axis 2 of en_t (the input shape is [1, 512, T]).

import litert_torch

litert_f0n_path = EXPORT_DIR / 'f0n_predictor.tflite'
FORCE_EXPORT    = True   # set False to reuse an existing .tflite

if not FORCE_EXPORT and litert_f0n_path.exists():
    sz = litert_f0n_path.stat().st_size / 1e6
    print(f'  ↷ {litert_f0n_path.name}  (skipped — {sz:.2f} MB)')
else:
    t0 = _t()
    try:
        edge_model = litert_torch.convert(
            f0n_lt_module.eval(),
            (en_t, style),
            dynamic_shapes=[{2: Dim('T_f0n', min=1, max=8192)}, None],
        )
        edge_model.export(str(litert_f0n_path))
        elapsed = (_t() - t0) * 1e3
        sz = litert_f0n_path.stat().st_size / 1e6
        print(f'  ✓ {litert_f0n_path.name}  {elapsed:.0f} ms  {sz:.2f} MB')
    except Exception as e:
        elapsed = (_t() - t0) * 1e3
        print(f'  ✗ {litert_f0n_path.name}  {elapsed:.0f} ms  — {e}')
        import traceback; traceback.print_exc()


/rhome/eingerman/Projects/DeepLearning/TTS/Kokoro/.venv/lib/python3.12/site-packages/torch/distributed/distributed_c10d.py:351: UserWarning: Device capability of jax unspecified, assuming `cpu` and `cuda`. Please specify it via the `devices` argument of `register_backend`.
  warnings.warn(


  ✗ f0n_predictor.tflite  8976 ms  — float32[1,512,-9223372036854775808]

While executing %sub : [num_users=2] = call_function[target=torch.ops.aten.sub.Tensor](args = (%en_t, %mean), kwargs = {})
Original traceback:
File "/tmp/ipykernel_2449000/4188570979.py", line 86, in forward
    F0 = block(F0, s)
  File "/rhome/eingerman/Projects/DeepLearning/TTS/Kokoro/kokoro/istftnet.py", line 443, in forward
    out = self._residual(x, s)
  File "/rhome/eingerman/Projects/DeepLearning/TTS/Kokoro/kokoro/istftnet.py", line 90, in forward
    normalized = self.norm(x)
  File "/tmp/ipykernel_2449000/4188570979.py", line 47, in forward
    diff = x - mean                                 # [N, C, T]
Use tlparse to see full graph. (https://github.com/pytorch/tlparse?tab=readme-ov-file#tlparse-parse-structured-pt2-logs)


Traceback (most recent call last):
  File "/rhome/eingerman/Projects/DeepLearning/TTS/Kokoro/.venv/lib/python3.12/site-packages/litert_torch/odml_torch/jax_bridge/_wrap.py", line 89, in lower_wrapper
    return jaxfn(*jaxfn_args, **jaxfn_kwargs)
  File "/rhome/eingerman/Projects/DeepLearning/TTS/Kokoro/.venv/lib/python3.12/site-packages/litert_torch/odml_torch/lowerings/_jax_lowerings.py", line 560, in jax_lowering
    return jnp.subtract(self.astype(promoted_type), other.astype(promoted_type))
  File "/rhome/eingerman/Projects/DeepLearning/TTS/Kokoro/.venv/lib/python3.12/site-packages/jax/_src/numpy/ufunc_api.py", line 182, in __call__
    return call(*args)
jax._src.source_info_util.JaxStackTraceBeforeTransformation: AssertionError: float32[1,512,-9223372036854775808]

The preceding stack trace is the source of the JAX operation that, once transformed by JAX, triggered the following exception.

--------------------

The above exception was the direct cause of the following exception:

In [14]:
# ═══════════════════════════════════════════════════════════════════════════════
#  LiteRT Inference — compare to PyTorch reference
# ═══════════════════════════════════════════════════════════════════════════════

if not litert_f0n_path.exists():
    print('No .tflite file — run the export cell first.')
else:
    lt_f0n  = litert_torch.load(str(litert_f0n_path))

    # en_t is [1, 512, T] — same C-first layout as F0NPredictorModuleLT
    lt_out = lt_f0n(en_t.numpy(), style.numpy())

    F0_lt_inf = np.asarray(lt_out[0])   # [1, T_f0]
    N_lt_inf  = np.asarray(lt_out[1])   # [1, T_f0]

    f0_inf_diff = np.abs(to_numpy(F0_pred) - F0_lt_inf).max()
    n_inf_diff  = np.abs(to_numpy(N_pred)  - N_lt_inf).max()
    print('LiteRT vs PyTorch max diff:')
    print(f'  F0_pred: {f0_inf_diff:.2e}')
    print(f'  N_pred:  {n_inf_diff:.2e}')
    print(f'  shapes  → F0: {F0_lt_inf.shape}  N: {N_lt_inf.shape}')


No .tflite file — run the export cell first.
